# Exploitation de données électorales avec Python
**Evaluation intermédiaire Python pour la data science (mi-semestre 2026)**

---

## Import des données

In [ ]:
import pandas as pd
from great_tables import GT

df = pd.read_csv(
    'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

df.head(10)

---
## 1. Explorations générales

### Question 1

Créer ou mettre à jour les variables suivantes :
- **`code_commune`** : En utilisant la variable déjà existante et le département, remplacer la valeur `code_commune` pour constituer un vrai code commune. Par exemple, pour Montrouge, vous devriez obtenir `92049`.
- **`candidat`** : créer une colonne avec le prénom et le nom mis ensemble, en n'oubliant pas de mettre un espace. Ne pas éliminer les bulletins abstentions, blancs ou nuls.

In [ ]:
# Question 1 : création de code_commune et candidat

# Créer code_commune complet (département + commune sur 3 chiffres)
df["code_commune"] = df["code_departement"].astype(str) + df["code_commune"].astype(str).str.zfill(3)

# Créer la colonne candidat (prenom + nom)
df["candidat"] = df["prenom"].fillna("") + " " + df["nom"].fillna("")
df["candidat"] = df["candidat"].str.strip()  # supprime les espaces inutiles

# Vérification sans afficher les index
df[["code_departement","code_commune","prenom","nom","candidat"]].head(10).style.hide(axis="index")


---
### Question 2

Compléter la phrase suivante grâce à Python :

> En 2022, il y avait XXXXX candidats à l'élection présidentielle

⚠️ Attention aux votes non exprimés et aux abstentions.

Utiliser cette f-string comme réponse :
```python
f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."
```

In [ ]:
# Question 2
df['candidat'].unique() #cette ligne indique
candidats = df['candidat'].dropna().nunique()

print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")

---
### Question 3

Calculer les scores nationaux de chaque candidat. Représenter dans un tableau, pour chaque candidat, le **nombre de voix** et le **pourcentage des votes exprimés** (c'est-à-dire en retirant abstentions et votes non exprimés).

Représenter cela dans un dataframe ou, pour avoir tous les points, dans un tableau mis en forme via `great_tables`.

In [ ]:
# Question 3 : scores nationaux corrigés avec % affiché

exclure = ["", "abstentions", "blancs", "nuls"]
votes_exprimes = df[~df["candidat"].isin(exclure)]

scores = votes_exprimes.groupby("candidat")["voix"].sum().reset_index()
total_votes_exprimes = votes_exprimes["voix"].sum()

# Calcul du pourcentage et ajout du symbole %
scores["Score (% votes exprimés)"] = (scores["voix"] / total_votes_exprimes * 100).round(2).map(lambda x: f"{x:.2f} %")

# Renommer les colonnes
scores.rename(columns={
    "candidat": "Candidat",
    "voix": "Nombre votes (total)"
}, inplace=True)

# Trier du plus grand au plus petit
scores = scores.sort_values(by="Nombre votes (total)", ascending=False).reset_index(drop=True)

# Affichage sans index
scores.style.hide(axis="index")
(
    GT(scores)
    .tab_header(title="Résultats du premier tour 2022")
    .cols_label(**{"Candidat": "Candidat", "Nombre votes (total)": "Votes", "Score (% votes exprimés)": "Score (%)"})
    .fmt_number(columns="Nombre votes (total)", decimals=0, sep_mark=" ")
)

---
## 2. Comparaison des scores départements aux moyennes nationales

### Question 4

Créer un dataframe nommé `score_departements` stockant, pour chaque département, le **nombre de votes** obtenu pour chaque candidat et le **score (en %)**.

In [ ]:
# Question 4

score_departements = (
    votes_exprimes
    .groupby(['code_departement', 'candidat'], as_index=False)['voix']
    .sum()
    .rename(columns={'voix': 'votes_departement'})
)

total_dep = (
    votes_exprimes
    .groupby('code_departement')['voix']
    .sum()
    .rename('total_dep')
)

score_departements = score_departements.merge(total_dep, on='code_departement')
score_departements['score_departement'] = score_departements['votes_departement'] / score_departements['total_dep'] * 100
score_departements = score_departements.drop(columns='total_dep')

# Vérification département 11
(
    GT(score_departements[score_departements['code_departement'] == '11'])
    .tab_header(title="Résultats par département - Aude (11)")
    .cols_label(code_departement="Département", candidat="Candidat", votes_departement="Votes", score_departement="Score (%)")
    .fmt_number(columns="votes_departement", decimals=0, sep_mark=" ")
    .fmt_number(columns="score_departement", decimals=2)
)

In [ ]:
# Vérification : résultat pour l'Aude (département n°11)
score_departements[score_departements['code_departement'] == '11']

---
### Question 5

Refaire le lien avec le niveau national pour comparer le score départemental avec le score national. Nommer ce dataframe `score_departements` (nous allons le réutiliser par la suite).

In [ ]:
# Question 5 : comparaison score départemental vs score national

# On récupère les scores nationaux dans un dictionnaire pour les fusionner
scores_nationaux_dict = scores.set_index("Candidat")["Score (% votes exprimés)"].str.replace(" %", "").astype(float).to_dict()

# Ajouter le score national à chaque ligne du score_departements
score_departements["score_national"] = score_departements["candidat"].map(scores_nationaux_dict)

# Calcul de l'écart entre score départemental et score national
score_departements["ecart_national"] = score_departements["score_departement"] - score_departements["score_national"]


In [ ]:
# Vérification : résultat pour l'Aude (département n°11)
(
    GT(score_departements[score_departements['code_departement'] == '11'])
    .tab_header(title="Résultats - Aude (11)")
    .fmt_number(columns=["votes_departement", "votes_national"], decimals=0, sep_mark=" ")
    .fmt_number(columns=["score_departement", "score_national"], decimals=2)
)

---
### Question 6

Créer une variable **`surrepresentation`** qui compare, en relatif, les scores nationaux et départementaux.

Par exemple, si un candidat a un score de 30% dans un département mais de 15% ailleurs, la valeur de `surrepresentation` sera égale à **100 (%)**.

In [ ]:
# Question 6

score_departements['surrepresentation'] = (
    (score_departements['score_departement'] - score_departements['score_national'])
    / score_departements['score_national']
    * 100
)

---
### Question 7

Créer une **fonction** pour représenter une figure des principales surreprésentations (en valeur absolue) par département, pour un candidat donné.

In [ ]:
# Question 7


---
## 3. Un peu de cartographie

Récupération du fond de carte des départements :

In [ ]:
from cartiflette import carti_download

departement_borders = carti_download(
    values = ["France"],
    crs = 4326,
    borders = "DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

---
### Question 8

Faire une **fonction** permettant de restreindre `score_departements` en fonction d'un candidat. Commencer par tester sur **Marine Le Pen** (créer un nouvel objet, ne pas écraser `score_departements`).

Faire une jointure au fond de carte des départements et effectuer une **carte de la surreprésentation**.

In [ ]:
# Question 8
from cartiflette import carti_download
import geopandas as gpd
import matplotlib.pyplot as plt

departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

def carte_candidat(candidat):
    df_candidat = score_departements[score_departements['candidat'] == candidat].copy()
    
    gdf = departement_borders.merge(
        df_candidat,
        left_on='DEP',
        right_on='code_departement',
        how='left'
    )
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    gdf.plot(
        column='surrepresentation',
        ax=ax,
        legend=True,
        cmap='RdBu',
        legend_kwds={'label': 'Surreprésentation (%)'},
        missing_kwds={'color': 'lightgrey'}
    )
    ax.set_title(f"Surreprésentation de {candidat} par département", fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Test sur Marine Le Pen
carte_candidat('Marine LE PEN')